# 機能情報を使った創薬標的ランキング（知名度の偏りを抑える）

これまでの検証で、スコアの多くが「遺伝子の知名度」と結び付いていることが課題でした。このノートブックでは、**遺伝子の機能情報をプロンプトに入れた判断（M3sF）と、入れない判断（M3s）を平均する**ことで、知名度に頼らない判断を加えます。

| 段階 | 中身 | 使い方 |
|---|---|---|
| 1. 採点（全遺伝子） | M3s（ノートブック 03 と同じ）と M3sF（遺伝子行の後に機能情報を3行足す）を両順で聞き、対数オッズを平均する | **最終順位** |
| 2. 確認（上位 `TOP_K2`） | 段階1の上位を、5択 × スイス式 × Bradley-Terry（ノートブック 04 と同じ）で並べ直す | 「別の聞き方でも上位か」の確認。順位は変えない |
| 3. 理由のラベル（上位 `N_LABEL`） | H3（直接効く）・R1（代償経路）・B1（機構が一致）を別々に聞き、上位の中で特に強い条件をラベルにする | 解釈用。順位には使わない |

## 機能情報（`scripts/fetch_function.py` → `data/genes/function_cache.json`）
UniProt の Function（最初の2文）、GO の分子機能・生物学的プロセス、Reactome の経路名を、遺伝子行の後に3行で入れます。

```
Gene: DHFR (Dihydrofolate reductase)
Function: Catalyzes the reduction of 7,8-dihydrofolate (DHF) to 5,6,7,8-tetrahydrofolate ... Key enzyme in folate metabolism.
GO: dihydrofolate reductase activity; folic acid binding; ...
Pathways: G1/S-Specific Transcription; Metabolism of folate and pterines; ...
```

**答えを教えないための対策**（症状の説明文に分子名を書かないのと同じ考え方）：UniProt の「Involvement in disease」欄は取得しない／disease・cancer・arthritis・schizophren・patient・mutation などを含む文や名前は丸ごと除く（例：TNF の「関節リウマチ患者の Treg の働きを弱める」という文）／最後に登録疾患の病名が残っていないことを自動で確認する。

## 実機検証の結果（txgemma-9b-chat Q6_K、5疾患 × 100遺伝子）
主な指標は、目的が新しい候補の発見なので「候補 vs ダミー」です。**知名度をそろえる**とは、C1（病気と関係なく「よく研究された遺伝子か」を聞いた点数）で全遺伝子を5つの層に分け、同じ層の中だけで比べることです（`scripts/eval06.py`）。

| 点数 | 候補 vs ダミー | **候補 vs ダミー（知名度をそろえる）** | 既知 vs その他 | 既知 vs ダミー（知名度をそろえる） | ダミーの Yes 率 |
|---|---|---|---|---|---|
| M3s（ノートブック 03） | 0.909 | 0.863 | 0.868 | 0.830 | 0.25 |
| M3sF（機能情報つき） | 0.927 | 0.829 | 0.864 | 0.868 | 0.07 |
| **M3s と M3sF の平均（このノートブック）** | **0.945** | **0.902** | **0.872** | **0.894** | — |
| 段階1と段階2の順位の悪い方 | 0.938 | 0.891 | 0.868 | **0.904** | — |
| （参考）C1 で M3s を補正した残差 | 0.663 | 0.794 | 0.783 | 0.833 | — |

- **機能情報だけでは、良くなる疾患と悪くなる疾患があります。** シスチン尿症（同じ SLC ファミリーのダミーとの見分け 0.667 → 0.889）や、GH1（14 → 8 位）・CYP17A1（22 → 10 位）・MS4A1（83 → 66 位）のように機能欄が治療の仕組みと合う遺伝子は上がりました。一方、FOLH1（47 → 78 位。放射性医薬品の標的だが機能欄は「葉酸分解酵素」）や統合失調症の受容体は下がりました。モデルは書かれた機能を強く信じるので、機能欄の書き方に左右されます。
- **M3s と平均すると、両者の弱点が打ち消し合って主な指標がすべて最良**になりました。
- **知名度を差し引く方法（残差）は逆効果**でした。本当の標的はよく研究されている遺伝子でもあるため、知名度と一緒に本当のシグナルまで消えます。M3s はもともと知名度をそろえても 0.863 を保っており、知名度だけ（C1）の 0.629 よりずっと高い＝**判断の大半は知名度ではない**ことも分かりました。
- 段階2（上位の再ランキング）は主な指標を上げなかったので、順位には使わず「別の聞き方でも上位か」の確認にとどめます。

## 前提・注意
- GGUF（llama-cpp-python）で動かします。100遺伝子で段階1が約12分、段階2が約2分、段階3が約1分です（txgemma-9b Q6_K、Mac）。モデルが無い環境ではモック（擬似乱数）です。
- 質問文・前置き・組み方は `scripts/yesno_question_variants.py`（M3s・H3・R1・B1）、`scripts/rank_variants.py`（M2r）、`scripts/swiss.py`、`scripts/pipeline06.py` と同じです（変えたら検証し直してください）。
- p（Yes の確率）はダミーでも高めに出るので、絶対基準（0.5 以上なら標的など）には使わず、**順位で判断**してください。

### このセルがすること：準備

In [1]:
import os, re, math, glob, json, time, random, hashlib, subprocess, sys
import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
ROOT = os.path.abspath("..")
print("ready:", ROOT)

ready: /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02


## 疾患の選択

### このセルがすること：`data/diseases.json` の登録疾患から1つ選び、病名・症状の箇条書き・遺伝子リストを読み込む

In [2]:
DISEASE_KEY = "ra"                    # "ra" / "scz" / "cystinuria" / "prostate_cancer" / "achondroplasia"
GENE_SET = "set100"

REGISTRY = json.load(open(os.path.join(ROOT, "data", "diseases.json"), encoding="utf-8"))
D = REGISTRY[DISEASE_KEY]
DISEASE, DISEASE_INFO = D["name"], D["info"][:5]
GENE_FILE = os.path.join(ROOT, "data", "genes", f"{D['gene_prefix']}_{GENE_SET}.tsv")
SET_NAME = os.path.basename(GENE_FILE).replace(".tsv", "")
print("選択:", DISEASE, "| 遺伝子リスト:", os.path.basename(GENE_FILE))

選択: rheumatoid arthritis | 遺伝子リスト: ra_set100.tsv


## 設定

### このセルがすること：段階2・3の対象数とモデルを決める
- `TOP_K2`：段階2（確認）にかける上位の数。`N_LABEL`：段階3（理由のラベル）を付ける上位の数。
- モデルは `~/llm/models` の GGUF を使います（前置きの KV キャッシュを使うため。Ollama の経路はこのノートブックにはありません）。

In [3]:
TOP_K2 = 40
N_LABEL = 20
N_ROUNDS, RANDOM_ROUNDS, JITTER, SEED = 8, 2, 0.3, 0   # 段階2（スイス式）の設定。検証と同じ値
N_CTX = 2048
MODEL_DIR = os.path.expanduser("~/llm/models")
ggufs = sorted(glob.glob(os.path.join(MODEL_DIR, "**", "*txgemma*.gguf"), recursive=True)) or sorted(glob.glob(os.path.join(MODEL_DIR, "**", "*.gguf"), recursive=True))
MODEL_PATH = ggufs[0] if ggufs else None
USE_LLM = MODEL_PATH is not None
print("モデル:", MODEL_PATH or "なし → モック（擬似乱数。数値に意味なし）")
OUT_DIR = os.path.join(ROOT, "outputs"); os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, f"{SET_NAME}_ranking06.csv")

モデル: /Users/yoshinorisatomi/llm/models/txgemma-9b-chat-Q6_K.gguf


## 入力遺伝子と機能情報

### このセルがすること：遺伝子リストと機能情報を読む（機能情報が無ければ `scripts/fetch_function.py` を実行して UniProt から取得する）
- 取得スクリプトは、疾患に触れる記述を除き、病名が残っていないことを確認してから保存します。

In [4]:
genes = pd.read_csv(GENE_FILE, sep="\t", dtype=str).fillna("")
genes["gene_label"] = [f"{g['symbol']} ({(g['protein_name_uniprot'] or g['gene_name']).split('|')[0]})" if (g["protein_name_uniprot"] or g["gene_name"]) else g["symbol"] for _, g in genes.iterrows()]
FCACHE = os.path.join(ROOT, "data", "genes", "function_cache.json")
if not os.path.exists(FCACHE):
    subprocess.run([sys.executable, os.path.join(ROOT, "scripts", "fetch_function.py")], check=True)
fcache = json.load(open(FCACHE))
genes["func_text"] = [fcache.get(u.split("|")[0], {}).get("text", "") for u in genes["uniprot_ids"]]
names = [v["name"].lower() for v in REGISTRY.values()]
assert not any(n in t.lower() for t in genes["func_text"] for n in names), "機能情報に病名が含まれています"
print(len(genes), "genes | 機能情報あり:", int((genes["func_text"] != "").sum()))
print(genes.loc[0, "gene_label"]); print(genes.loc[0, "func_text"])

100 genes | 機能情報あり: 100
PADI4 (peptidyl arginine deiminase 4)
Function: Citrullinates EP300/P300 at 'Arg-2142', which favors its interaction with NCOA2/GRIP1
GO: calcium ion binding; identical protein binding; chromatin organization; chromatin remodeling; negative regulation of heterochromatin formation; neutrophil extracellular trap formation
Pathways: Chromatin modifying enzymes



## 質問とプロンプト

### このセルがすること：Yes/No（M3s・H3・R1・B1）と5択（M2r）の質問・前置きを定義する（文面は検証スクリプトからそのまま）

In [5]:
QUESTIONS = {
 "M3s": "Consider these three criteria:\n(a) Inhibiting or activating {gene} could plausibly treat {disease} or improve at least one of the symptoms listed above.\n(b) Even outside the causal pathway, modulating {gene} could counteract the abnormal process described above through a parallel or opposing pathway.\n(c) {gene}'s own substrate, ligand, pathway, cell type or circuit precisely matches the mechanism of {disease} described above (a similar but distinct role, even in the same gene family, does not count).\nDoes this gene meet at least one of these criteria?",
 "H3": "Can a hypothesis be formulated that a therapeutic drug targeting the gene {gene} would treat {disease} or the symptoms of {disease}?",
 "R1": "Even if this gene is not part of the pathway that causes {disease}, could activating or inhibiting it counteract or compensate for the abnormal process described above, for example through a parallel or opposing pathway in the same cells?",
 "B1": "Does this gene's own specific role — its substrate, ligand, signalling pathway, cell type or circuit — match the mechanism described above precisely, rather than a related but distinct one (e.g. a different molecule, cell type, tissue or subcellular compartment)? Answer No for a similar-sounding but distinct role, even in the same gene family."
}
YN_NOTE = 'Note: the vast majority of human genes are NOT drug targets for any given disease. Answer Yes only when there is clear evidence or a clear mechanistic link; otherwise answer No. Membership in the same gene family, superfamily or protein class as a true disease gene (e.g. being another member of the same transporter, channel, receptor or enzyme family) is NOT sufficient evidence by itself. Judge each gene on whether ITS OWN specific substrate, ligand, cargo or interaction partner matches the mechanism described above, not on family resemblance alone.\n'
CHOICE_Q = "Consider these three criteria:\n(a) A hypothesis can be constructed that inhibiting or activating the gene would treat {disease} or improve at least one of the symptoms listed above.\n(b) Even if the gene is not part of the pathway that causes {disease}, activating or inhibiting it could counteract or compensate for the abnormal process described above, for example through a parallel or opposing pathway in the same cells.\n(c) The gene's own specific role — its substrate, ligand, signalling pathway, cell type or circuit — matches the mechanism described above precisely, rather than a related but distinct one (e.g. a different molecule, cell type, tissue or subcellular compartment); a similar-sounding but distinct role, even in the same gene family, does not count.\nWhich numbered gene above best meets at least one of these criteria?"
CHOICE_NOTE = 'Note: the vast majority of human genes are NOT drug targets for any given disease. Pick a gene only when there is a clear, specific reason; membership in the same gene family as a known disease gene is NOT sufficient by itself.\n'
GROUP_SIZE = 5

def bullets(): return "\n".join(f"- {b}" for b in DISEASE_INFO)

def yn_prefix(order):
    options = "Answer each question with Yes or No." if order == "yes_first" else "Answer each question with No or Yes."
    return ("You are an expert in drug discovery and human disease biology.\n" + YN_NOTE +
            f"Disease: {DISEASE}\nTarget symptoms and the organ, cell and functional abnormalities behind them:\n{bullets()}\n{options}\n\n")

def q_line(qid, symbol):
    return f"{qid}. {QUESTIONS[qid].format(disease=DISEASE, gene=symbol)} Answer:"

def choice_prefix():
    return ("You are an expert in drug discovery and human disease biology. You will be shown a numbered list of "
            f"{GROUP_SIZE} candidate genes for one disease, and asked which ONE number is the best answer to a question.\n" + CHOICE_NOTE +
            f"Disease: {DISEASE}\nTarget symptoms and the organ, cell and functional abnormalities behind them:\n{bullets()}\n"
            f"Answer with a single number from 1 to {GROUP_SIZE + 1} only. No words, no explanation.\n\n")

def choice_block(labels):
    lines = [f"{i + 1}. {g}" for i, g in enumerate(labels)] + [f"{len(labels) + 1}. None of the above genes seem clearly relevant"]
    return "Candidate genes:\n" + "\n".join(lines) + "\n"

CHOICE_LINE = f"Question: {CHOICE_Q.format(disease=DISEASE)} Answer: "      # 末尾の空白が必要（数字はこの後に来る）
g0 = genes.iloc[0]
print(yn_prefix("yes_first") + f"Gene: {g0['gene_label']}\n" + g0["func_text"] + q_line("M3s", g0["symbol"]))

You are an expert in drug discovery and human disease biology.
Note: the vast majority of human genes are NOT drug targets for any given disease. Answer Yes only when there is clear evidence or a clear mechanistic link; otherwise answer No. Membership in the same gene family, superfamily or protein class as a true disease gene (e.g. being another member of the same transporter, channel, receptor or enzyme family) is NOT sufficient evidence by itself. Judge each gene on whether ITS OWN specific substrate, ligand, cargo or interaction partner matches the mechanism described above, not on family resemblance alone.
Disease: rheumatoid arthritis
Target symptoms and the organ, cell and functional abnormalities behind them:
- Joint pain, swelling and morning stiffness, symmetric in the small joints of hands and feet: caused by chronic inflammation of the synovial membrane that lines the joints
- Progressive joint deformity and loss of function: invasive growth of the synovial lining cells and

## エンジン

### `yes_probs(label, extra, symbol, qids, order)` — 1遺伝子に複数の Yes/No 質問を独立に聞く
前置き（order ごとに保存）を復元 → 遺伝子行（`extra` に機能情報）を処理 → 質問ごとにその位置まで巻き戻して処理し、Yes の確率を読む。
### `choice_probs(labels)` — 5遺伝子＋該当なしから選ばれる確率
前置きを復元 → 番号付きリスト → 質問 → 「Answer: 」の直後の数字 1〜6 の確率。

In [6]:
def softmax(l):
    l = np.asarray(l, float); p = np.exp(l - l.max()); return p / p.sum()

if USE_LLM:
    from llama_cpp import Llama
    llm = Llama(model_path=MODEL_PATH, n_ctx=N_CTX, n_gpu_layers=-1, logits_all=False, verbose=False)
    def tok(text, bos=False): return llm.tokenize(text.encode("utf-8"), add_bos=bos, special=bos)
    def ids_of(sp):
        out = []
        for s in sp:
            t = tok(s)
            if len(t) == 1 and t[0] not in out: out.append(t[0])
        return out
    YES_IDS = ids_of(["Yes", " Yes", "yes", " yes", "YES", " YES"]); NO_IDS = ids_of(["No", " No", "no", " no", "NO", " NO"])
    DIGIT = {n: tok(str(n)) for n in range(1, GROUP_SIZE + 2)}
    STATE = {}
    for name, text in (("yes_first", yn_prefix("yes_first")), ("no_first", yn_prefix("no_first")), ("choice", choice_prefix())):
        llm.reset(); llm.eval(tok(text, bos=True)); STATE[name] = llm.save_state()
    def logits():
        return np.ctypeslib.as_array(llm._ctx.get_logits(), shape=(llm.n_vocab(),)).astype(np.float64)
    def lse(v): m = max(v); return m + math.log(sum(math.exp(x - m) for x in v))
    def yes_probs(label, extra, symbol, qids, order):
        llm.reset(); llm.load_state(STATE[order]); llm.eval(tok(f"Gene: {label}\n" + extra)); base = llm.n_tokens
        out = {}
        for q in qids:
            llm.n_tokens = base; llm.eval(tok(q_line(q, symbol)))           # 質問ごとに遺伝子行の直後へ巻き戻す（独立に聞く）
            lg = logits(); lg -= lg.max(); lp = lg - math.log(np.exp(lg).sum())
            out[q] = 1 / (1 + math.exp(lse([lp[i] for i in NO_IDS]) - lse([lp[i] for i in YES_IDS])))
        return out
    def choice_probs(labels):
        llm.reset(); llm.load_state(STATE["choice"]); llm.eval(tok(choice_block(labels))); llm.eval(tok(CHOICE_LINE))
        lg = logits(); return softmax([lg[DIGIT[n][0]] for n in range(1, GROUP_SIZE + 2)])
else:
    def yes_probs(label, extra, symbol, qids, order):
        return {q: int(hashlib.md5((label + extra[:20] + q + order).encode()).hexdigest(), 16) % 1000 / 1000 for q in qids}
    def choice_probs(labels):
        return softmax([int(hashlib.md5(g.encode()).hexdigest(), 16) % 1000 / 250 for g in labels + ["NONE"]])
    print("警告: モックです。")
logit = lambda x: math.log(max(x, 1e-6) / max(1 - x, 1e-6))
t0 = time.time(); print({o: {k: round(v, 3) for k, v in yes_probs(g0["gene_label"], g0["func_text"], g0["symbol"], ["M3s"], o).items()} for o in ("yes_first", "no_first")}, f"{time.time() - t0:.1f}s")

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


{'yes_first': {'M3s': 0.236}, 'no_first': {'M3s': 0.113}} 3.0s


## 段階1：採点（全遺伝子）

### このセルがすること：M3s（機能情報なし）と M3sF（機能情報つき）を両順で聞き、対数オッズの平均を出す
- `m3s` / `m3sf`：両順の対数オッズの平均。`score`：その2つの平均（**最終順位はこれ**）。

In [7]:
rows, t0 = [], time.time()
for i, g in genes.iterrows():
    r = {"symbol": g["symbol"]}
    for col, extra in (("m3s", ""), ("m3sf", g["func_text"])):
        r[col] = float(np.mean([logit(yes_probs(g["gene_label"], extra, g["symbol"], ["M3s"], o)["M3s"]) for o in ("yes_first", "no_first")]))
    rows.append(r)
    if (i + 1) % 20 == 0: print(f"{i + 1}/{len(genes)} ({time.time() - t0:.0f}s)")
res = genes[["symbol", "gene_label", "category", "label"]].merge(pd.DataFrame(rows), on="symbol")
res["score"] = (res["m3s"] + res["m3sf"]) / 2
res["rank"] = res["score"].rank(ascending=False, method="first").astype(int)
res = res.sort_values("rank").reset_index(drop=True)
print(f"段階1: {time.time() - t0:.0f}s")
res.head(10)[["rank", "symbol", "category", "score", "m3s", "m3sf"]].round(2)

20/100 (121s)


40/100 (242s)


60/100 (359s)


80/100 (472s)


100/100 (587s)
段階1: 587s


,rank,symbol,category,score,m3s,m3sf
0,1,TNF,known,10.29,10.21,10.37
1,2,IL1B,candidate,8.59,7.32,9.86
2,3,IL6,known,8.31,7.83,8.79
3,4,IL1R1,known,7.64,6.75,8.53
4,5,TNFRSF1A,candidate,6.63,6.31,6.95
5,6,JAK1,known,6.19,6.36,6.03
6,7,JAK2,known,5.74,5.88,5.61
7,8,IL17RA,candidate,5.72,5.28,6.16
8,9,JAK3,known,5.15,4.93,5.38
9,10,IL17A,candidate,5.15,5.08,5.23


## 段階2：確認（上位 `TOP_K2`、5択 × スイス式 × Bradley-Terry）

### このセルがすること：段階1の上位をノートブック 04 と同じ方法で並べ直し、「別の聞き方でも上位か」を出す（最終順位は変えない）
- `check_rank`：段階2での順位。`both_top`：段階1と段階2の順位の悪い方（両方で上位なら小さい）。

In [8]:
def random_groups(idx, rng):
    """rank_variants.py と同じ手順（シャッフル → 先頭から5つずつ → 端数は他から補充）。"""
    order = idx[:]; rng.shuffle(order); out = []
    for start in range(0, len(order), GROUP_SIZE):
        chunk = order[start:start + GROUP_SIZE]
        if len(chunk) < GROUP_SIZE: chunk = chunk + rng.sample([i for i in idx if i not in chunk], GROUP_SIZE - len(chunk))
        out.append(chunk)
    return out


def swiss_groups(idx, strength, rng, jitter):
    """強さの順に並べて先頭から5つずつ組む。強さに乱数（標準偏差 jitter × 強さの標準偏差）を足して毎回少しずらす。
    グループ内の並び（＝番号）はシャッフルして、番号の癖が強さと結び付かないようにする。"""
    s = np.array([strength[i] for i in idx]); s = s + np.array([rng.gauss(0, 1) for _ in idx]) * jitter * (s.std() or 1)
    order = [idx[k] for k in np.argsort(-s)]; out = []
    for start in range(0, len(order), GROUP_SIZE):
        chunk = order[start:start + GROUP_SIZE]
        if len(chunk) < GROUP_SIZE: chunk = chunk + order[start - (GROUP_SIZE - len(chunk)):start]      # 端数は直前（強さの近い）遺伝子で補充
        chunk = chunk[:]; rng.shuffle(chunk); out.append(chunk)
    return out


def luce(groups, n, iters=1000, prior=0.5):
    """Luce の選択モデル（Bradley-Terry を多者択一に広げたもの）の強さを MM 法（Hunter 2004）で推定する。
    groups: [(メンバーの番号の並び, 各メンバーが選ばれた確率)]。遺伝子 i が選ばれる確率 = π_i / Σ_{グループ内} π。
    各項目に「強さ1の仮想相手との prior 勝ち・prior 負け」を足して、全勝・全敗でも発散しないようにする。return: log π"""
    W, pi = np.full(n, prior), np.ones(n)
    for mem, p in groups:
        for m, q in zip(mem, p): W[m] += q
    for _ in range(iters):
        den = 2 * prior / (pi + 1)
        for mem, p in groups:
            den[mem] += 1 / pi[mem].sum()
        new = W / den; new /= np.exp(np.mean(np.log(new)))
        if np.max(np.abs(np.log(new) - np.log(pi))) < 1e-10: pi = new; break
        pi = new
    return np.log(pi)


top = res.head(TOP_K2).index.tolist()
rng, rows2, t0 = random.Random(SEED), [], time.time()
def fit():
    items = top + ["NONE"]; ix = {s: k for k, s in enumerate(items)}; groups = []
    for _, d in pd.DataFrame(rows2).groupby(["round", "group"]):
        d = d.sort_values("position")
        groups.append(([ix[i] for i in d["gi"]] + [ix["NONE"]], np.r_[d["p"].values, d["p_none"].iloc[0]]))
    return luce(groups, len(items))
for r in range(N_ROUNDS):
    groups = random_groups(top, rng) if r < RANDOM_ROUNDS else swiss_groups(top, dict(zip(top, fit())), rng, JITTER)
    for gi, chunk in enumerate(groups):
        p = choice_probs([res.at[i, "gene_label"] for i in chunk])
        for slot, i in enumerate(chunk):
            rows2.append({"round": r, "group": gi, "position": slot + 1, "gi": i, "p": round(float(p[slot]), 6), "p_none": round(float(p[-1]), 6)})
s = fit()
res["check_bt"] = pd.Series(s[:-1], index=top)
res["check_rank"] = res["check_bt"].rank(ascending=False, method="first")
res["both_top"] = np.maximum(res["rank"], res["check_rank"])
print(f"段階2: {time.time() - t0:.0f}s | 段階1と段階2の順位相関（上位 {TOP_K2}） {res.loc[top, 'rank'].corr(res.loc[top, 'check_rank'], method='spearman'):.2f}")

段階2: 119s | 段階1と段階2の順位相関（上位 40） 0.53


## 段階3：理由のラベル（上位 `N_LABEL`）

### このセルがすること：H3（直接効く）・R1（代償経路）・B1（機構が一致）を別々に聞き、上位の中で特に強い条件をラベルにする
- 上位の遺伝子は H3・R1 がほぼ全員 Yes、B1 はほぼ全員 No になるので、Yes/No そのままでは区別が付きません。そこで、各条件の対数オッズを上位 `N_LABEL` の中で標準化し、**平均より 0.5 標準偏差以上高い条件**をラベルにします（無ければ最も高い条件）。
- 機能情報は入れずに聞きます（検証と同じ）。順位には使いません。

In [9]:
LAB = {"H3": "直接効く", "R1": "代償経路", "B1": "機構が一致"}
lab_idx = res.head(N_LABEL).index
t0 = time.time()
for i in lab_idx:
    g = res.loc[i]
    ps = {o: yes_probs(g["gene_label"], "", g["symbol"], list(LAB), o) for o in ("yes_first", "no_first")}
    for q in LAB: res.at[i, q] = float(np.mean([logit(ps[o][q]) for o in ps]))
zs = (res.loc[lab_idx, list(LAB)] - res.loc[lab_idx, list(LAB)].mean()) / res.loc[lab_idx, list(LAB)].std()
res["reasons"] = ""
for i in lab_idx:
    strong = [LAB[q] for q in LAB if zs.at[i, q] >= 0.5]
    res.at[i, "reasons"] = "・".join(strong) if strong else LAB[zs.loc[i].idxmax()]
res.round(4).to_csv(OUT_CSV, index=False)
print(f"段階3: {time.time() - t0:.0f}s | wrote {OUT_CSV}")
res.head(N_LABEL)[["rank", "symbol", "category", "score", "check_rank", "both_top", "reasons"] + list(LAB)].round(2)

段階3: 61s | wrote /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02/outputs/ra_set100_ranking06.csv


,rank,symbol,category,score,check_rank,both_top,reasons,H3,R1,B1
0,1,TNF,known,10.29,3.0,3.0,直接効く・代償経路・機構が一致,12.94,8.40,4.18
1,2,IL1B,candidate,8.59,2.0,2.0,直接効く・機構が一致,10.21,4.69,-0.28
2,3,IL6,known,8.31,6.0,6.0,代償経路・機構が一致,9.31,5.18,0.28
3,4,IL1R1,known,7.64,1.0,4.0,直接効く・代償経路・機構が一致,9.66,5.22,0.17
4,5,TNFRSF1A,candidate,6.63,5.0,5.0,直接効く・代償経路・機構が一致,10.53,5.49,0.30
5,6,JAK1,known,6.19,17.0,17.0,直接効く,10.37,1.79,-2.75
6,7,JAK2,known,5.74,11.0,11.0,直接効く,9.75,1.76,-2.87
7,8,IL17RA,candidate,5.72,4.0,8.0,直接効く,9.51,3.25,-0.71
8,9,JAK3,known,5.15,25.0,25.0,直接効く,8.98,0.77,-3.43
9,10,IL17A,candidate,5.15,13.0,13.0,機構が一致,9.15,3.63,-0.82


## 評価

### このセルがすること：段階1の点数・M3s だけ・M3sF だけの AUC と、既知遺伝子の順位を出す
- 知名度をそろえた AUC は C1 の点数が必要なので、ここでは出しません（`scripts/eval06.py` を参照）。

In [10]:
def auc(pos, neg):
    pos, neg = np.asarray(list(pos), float), np.asarray(list(neg), float)
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    return float(((pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()) / (len(pos) * len(neg)))
kn, rd, cd = (res["category"] == c for c in ("known", "random", "candidate"))
display(pd.DataFrame({c: {"AUC 候補 vs ダミー": auc(res.loc[cd, c], res.loc[rd, c]), "AUC 既知 vs その他": auc(res.loc[kn, c], res.loc[~kn, c]),
                          "AUC 既知 vs ダミー": auc(res.loc[kn, c], res.loc[rd, c]), "ダミーの Yes 率（中央値）": float(np.median(1 / (1 + np.exp(-res.loc[rd, c]))))}
                      for c in ("score", "m3s", "m3sf")}).round(3))
print("既知遺伝子の順位:")
res.loc[kn, ["rank", "symbol", "score", "m3s", "m3sf", "check_rank", "reasons"]].round(2)

,score,m3s,m3sf
AUC 候補 vs ダミー,0.959,0.951,0.951
AUC 既知 vs その他,0.720,0.703,0.711
AUC 既知 vs ダミー,1.000,0.971,0.995
ダミーの Yes 率（中央値）,0.105,0.188,0.057


既知遺伝子の順位:


,rank,symbol,score,m3s,m3sf,check_rank,reasons
0,1,TNF,10.29,10.21,10.37,3.0,直接効く・代償経路・機構が一致
2,3,IL6,8.31,7.83,8.79,6.0,代償経路・機構が一致
3,4,IL1R1,7.64,6.75,8.53,1.0,直接効く・代償経路・機構が一致
5,6,JAK1,6.19,6.36,6.03,17.0,直接効く
6,7,JAK2,5.74,5.88,5.61,11.0,直接効く
8,9,JAK3,5.15,4.93,5.38,25.0,直接効く
20,21,IL6R,4.08,6.24,1.93,9.0,
21,22,PTGS2,4.07,3.63,4.51,21.0,
37,38,TNFSF11,2.98,2.52,3.44,18.0,
43,44,CD80,2.56,2.17,2.96,NaN,


## 対話型グラフ（ホバーで遺伝子名）

### このセルがすること：M3s × M3sF の散布図、上位の一覧（理由つき）、段階1と段階2の順位の比較を描く
- 同じ図を `outputs/<セット名>_ranking06_charts.html` に保存します（plotly.js 埋め込み、ネット接続なしで開けます）。

In [11]:
import plotly.graph_objects as go
CAT = [("known", "既知", "#2a78d6", "circle"), ("candidate", "候補", "#eb6834", "square"), ("random", "ダミー", "#1baf7a", "diamond")]
hover = "<b>%{customdata[0]}</b>（%{customdata[1]}）<br>順位 %{customdata[2]}<br>%{customdata[3]}<extra></extra>"
cd_ = lambda d: np.c_[d["symbol"], d["category"], d["rank"], d["reasons"].fillna("")]

fig = go.Figure()
for cat, jp, col, mk in CAT:
    d = res[res["category"] == cat]
    fig.add_trace(go.Scatter(x=d["m3s"], y=d["m3sf"], mode="markers", name=jp, customdata=cd_(d), hovertemplate=hover,
                             marker=dict(color=col, symbol=mk, size=9 if cat == "known" else 7, opacity=0.8, line=dict(width=1, color="white"))))
lo, hi = float(res[["m3s", "m3sf"]].min().min()), float(res[["m3s", "m3sf"]].max().max())
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", line=dict(color="#c3c2b7", dash="dot"), showlegend=False, hoverinfo="skip"))
fig.update_layout(title=f"{DISEASE}: M3s（機能情報なし）× M3sF（機能情報つき）。右上ほど上位、点線より上＝機能情報で上がった",
                  xaxis_title="M3s（対数オッズ）", yaxis_title="M3sF（対数オッズ）", template="plotly_white", width=800, height=600)
fig.show()

top_n = res.head(N_LABEL).iloc[::-1]
fig2 = go.Figure(go.Bar(x=top_n["score"], y=[f"{r} {s}" for r, s in zip(top_n["rank"], top_n["symbol"])], orientation="h",
                        marker_color=[dict((c, col) for c, _, col, _ in CAT)[c] for c in top_n["category"]],
                        text=top_n["reasons"], textposition="outside", customdata=cd_(top_n), hovertemplate=hover))
fig2.update_layout(title=f"上位 {N_LABEL}（点数と理由のラベル。青＝既知、橙＝候補、緑＝ダミー）", xaxis_title="段階1の点数", template="plotly_white",
                   width=900, height=24 * N_LABEL + 160, margin=dict(l=120))
fig2.show()

t2 = res.head(TOP_K2)
fig3 = go.Figure()
for cat, jp, col, mk in CAT:
    d = t2[t2["category"] == cat]
    fig3.add_trace(go.Scatter(x=d["rank"], y=d["check_rank"], mode="markers+text", name=jp, text=d["symbol"], textposition="top center",
                              textfont=dict(size=9), customdata=cd_(d), hovertemplate=hover,
                              marker=dict(color=col, symbol=mk, size=9, line=dict(width=1, color="white"))))
fig3.add_trace(go.Scatter(x=[1, TOP_K2], y=[1, TOP_K2], mode="lines", line=dict(color="#c3c2b7", dash="dot"), showlegend=False, hoverinfo="skip"))
fig3.update_layout(title=f"段階1の順位 × 段階2（5択・スイス式）の順位（左下＝両方で上位）", xaxis=dict(title="段階1の順位", autorange="reversed"),
                   yaxis=dict(title="段階2の順位", autorange="reversed"), template="plotly_white", width=800, height=600)
fig3.show()

html_path = OUT_CSV.replace(".csv", "_charts.html")
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'></head><body>")
    for i, fg in enumerate((fig, fig2, fig3)):
        f.write(fg.to_html(full_html=False, include_plotlyjs=True if i == 0 else False))
    f.write("</body></html>")
print("saved:", html_path)

saved: /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02/outputs/ra_set100_ranking06_charts.html
